In [1]:
!pip install transformers torch accelerate huggingface_hub -q


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import torch
from transformers import pipeline
from huggingface_hub import login

HF_TOKEN = os.getenv("HF_TOKEN", "ВСТАВЬТЕ_ВАШ_TOKEN_ЗДЕСЬ")
login(token=HF_TOKEN)

# Загружаем модель (Llama-3-8B-Instruct)
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

pipe = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)
print("✅ Модель Llama 3 успешно загружена и готова к работе!")

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:02<00:00, 98.21it/s] 


✅ Модель Llama 3 успешно загружена и готова к работе!


In [3]:
import json
import re

# 1. Текст для анализа
user_text = """
Настоящим я даю согласие на передачу моего ИИН в РОО НЦНЭ для целей обработки данных. 
Данные будут использованы для ведения статистики и отчетности. 
Пользователь уведомлен о рисках передачи персональных данных третьим лицам.
"""

# 2. Улучшенный промпт (строго на русском + формат)
prompt = f"""
Ты — профессиональный ИИ-юрист. Твоя задача — провести аудит текста.
ОТВЕЧАЙ ТОЛЬКО НА РУССКОМ ЯЗЫКЕ. Это критически важно.

Выведи результат СТРОГО в формате JSON с этими ключами: 
"organization", "privacy_score", "risk_level", "key_risks", "detailed_report", "verdict".

Текст для анализа:
{user_text}
"""

messages = [{"role": "user", "content": prompt}]

print("⏳ Анализирую и рисую отчет...")
outputs = pipe(messages, max_new_tokens=1024, temperature=0.1)
raw_content = outputs[0]["generated_text"][-1]["content"]

# Функция для создания визуальной шкалы
def get_visual_score(score):
    try:
        score = int(score)
        if score >= 80: return f"{score} 🟢 (Безопасно)"
        if score >= 50: return f"{score} 🟡 (Средний риск)"
        return f"{score} 🔴 (ОПАСНО)"
    except:
        return str(score)

try:
    match = re.search(r"\{.*\}", raw_content, re.DOTALL)
    if match:
        data = json.loads(match.group())
        
        # Добавляем смайлик и шкалу прямо в данные перед выводом
        score_val = data.get("privacy_score", 0)
        data["privacy_score_visual"] = get_visual_score(score_val)
        
        print("\n" + "="*40)
        print("🛡️ ОТЧЕТ БЕЗОПАСНОСТИ CONSENT OS")
        print("="*40)
        
        # Выводим самое важное крупно
        print(f"\nОРГАНИЗАЦИЯ: {data.get('organization')}")
        print(f"РЕЙТИНГ ПРИВАТНОСТИ: {data['privacy_score_visual']}")
        print(f"УРОВЕНЬ РИСКА: {data.get('risk_level').upper()}")
        print("-" * 20)
        
        print(f"КЛЮЧЕВЫЕ РИСКИ:")
        for risk in data.get("key_risks", []):
            print(f"  ⚠️ {risk}")
            
        print(f"\nПОДРОБНЫЙ ОТЧЕТ:\n{data.get('detailed_report')}")
        print(f"\nВЕРДИКТ: {data.get('verdict')}")
        print("\n" + "="*40)
        
    else:
        print("Ошибка: JSON не найден.")
except Exception as e:
    print(f"Ошибка парсинга: {e}")
    print("Сырой ответ модели:", raw_content)

⏳ Анализирую и рисую отчет...

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Both `max_new_tokens` (=1024) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🛡️ ОТЧЕТ БЕЗОПАСНОСТИ CONSENT OS

ОРГАНИЗАЦИЯ: РОО НЦНЭ
РЕЙТИНГ ПРИВАТНОСТИ: 60 🟡 (Средний риск)
УРОВЕНЬ РИСКА: MEDIUM
--------------------
КЛЮЧЕВЫЕ РИСКИ:
  ⚠️ Передача персональных данных третьим лицам

ПОДРОБНЫЙ ОТЧЕТ:
В тексте обнаружены следующие риски для приватности: передача ИИН третьим лицам. Пользователь уведомлен о рисках, но не предоставлен подробный список получателей данных. Ведение статистики и отчетности может привести к утечке персональных данных. 

ВЕРДИКТ: Рекомендуется уточнить получателей данных и обеспечить безопасность передачи персональных данных

